# NB03 — Descriptive Statistics and EDA for Finance

**Role:** Core  
**Manual section:** 2.2.2.4 — Statistical fundamentals for EDA  
**Prerequisites:** NB02

---

## Purpose of this notebook

This notebook is the bridge between Topic 1 and Topic 2. It introduces Exploratory Data Analysis (EDA) as a business diagnosis tool — the systematic inspection of data before any modelling begins.

**Dataset:** `dataset_B_descriptive_stats_eda.csv` (45 invoices × 10 columns)

**What you will learn:**
- How to detect missing values and assess data quality
- How to summarise central tendency and dispersion
- How to use histograms, box plots and scatter plots for diagnosis
- How to compare segments and interpret correlations
- How to write a short analyst memo summarising findings

## Why this notebook is central

Notebook 03 is the **bridge** between Topic 1 and Topic 2. It still feels accessible and descriptive, but it already introduces the way analysts think before modelling.

The repeated pattern is now fully visible:

- **Diagnose** — what does the dataset look like?
- **Transform** — create summaries, comparisons and plots
- **Validate** — are the patterns plausible and useful?
- **Justify** — why do these observations matter before modelling?

Students should read this notebook as their first example of **business diagnosis before modelling**.

---

## Section 1 — Loading and First Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Resolve data directory — works from the project root or notebooks/
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

# Load the invoices dataset
df = pd.read_csv(DATA_DIR / "dataset_B_descriptive_stats_eda.csv")

print("Dataset loaded.")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
df.head(10)

In [ ]:
df.info()

**Column guide**

| Column | Type | Description |
|--------|------|-------------|
| `invoice_id` | int | Unique invoice identifier |
| `date` | str | Invoice date (YYYY-MM-DD) |
| `month_name` | str | Calendar month |
| `customer_segment` | str | Customer label (A, B, C, D) |
| `invoice_paid` | str | Whether the invoice was paid (`yes` / `no`) |
| `vat_type` | str | VAT category (`standard` 21 % / `reduced` 10 %) |
| `vat_pct` | int | VAT percentage |
| `invoice_amount` | int | Net invoice amount (EUR) |
| `vat_amount` | float | VAT amount (EUR) — contains **3 missing values** |
| `total_amount` | float | Gross total (EUR) |

This is a small invoicing dataset for a company with four customer segments across two
months. Although it has only 45 rows, it is large enough to practise the full EDA workflow.

### First inspection as business diagnosis

The first rows and the column guide do more than identify technical fields. They help us ask the first business questions:

- What is the unit of observation?
- Which variables look monetary, categorical or operational?
- Which fields could later be used for grouping, filtering or prediction?

This is why early inspection is not a ritual. It is the first analytical reading of the dataset.

---

## Section 2 — Missing Values and Basic Data Quality

The first thing to check after loading a dataset: **is anything missing or suspicious?**

In [ ]:
# Count missing values per column
df.isna().sum()

We have **3 missing values** in `vat_amount`. Let's inspect those rows:

In [ ]:
# Show rows where vat_amount is missing
df[df["vat_amount"].isna()]

**Observations:**

- The missing VAT amounts correspond to invoices where `total_amount` is still recorded.
- We could potentially **recalculate** them: `vat_amount = invoice_amount × vat_pct / 100`.
- For now we will note the issue and decide how to handle it later.

Let's also check for duplicates and suspicious values:

In [ ]:
# Duplicates
print("Duplicate rows:", df.duplicated().sum())

# Value ranges
print()
print("invoice_amount range:", df["invoice_amount"].min(), "–", df["invoice_amount"].max())
print("vat_pct unique values:", sorted(df["vat_pct"].unique()))
print("invoice_paid unique values:", sorted(df["invoice_paid"].unique()))

> **Quality verdict:** the dataset is clean apart from 3 missing VAT amounts.
> No duplicates, no impossible values. We can proceed with analysis.

### Data quality is already a modelling issue

Missing values, duplicates and suspicious ranges are often presented as a purely technical cleaning task. In reality, they already influence modelling quality.

If a variable is inconsistent, later summaries become less trustworthy. If a field is missing in a structured way, it may create bias. If a value is impossible, a model may learn nonsense.

So even before any algorithm appears, quality checks already protect the future modelling workflow.

---

## Section 3 — Central Tendency

Central tendency answers: **"What is the typical value?"**

The two most common measures are:
- **Mean** — the arithmetic average (sensitive to extreme values).
- **Median** — the middle value when sorted (robust to extremes).

In [ ]:
# Mean and median for key numeric columns
for col in ["invoice_amount", "vat_amount", "total_amount"]:
    m = df[col].mean()
    med = df[col].median()
    print(f"{col:18s}  mean = {m:8.1f}   median = {med:8.1f}   gap = {m - med:+.1f}")

When the mean and median are close, the distribution is roughly **symmetric**.
When the mean is noticeably higher than the median, there may be a **right skew**
(a few large values pulling the average up).

In [ ]:
# Grouped mean — average invoice amount by customer segment
df.groupby("customer_segment")["invoice_amount"].agg(["mean", "median", "count"])

In [ ]:
# Mean invoice amount by payment status
df.groupby("invoice_paid")["invoice_amount"].agg(["mean", "median", "count"])

> **Interpretation:** compare the segments. Does any customer segment have a notably
> higher or lower typical invoice? Do unpaid invoices tend to be larger or smaller?

### Grouped summaries are business understanding

When we compare means, medians or counts across customer segments or payment status, we are no longer just describing a table. We are asking whether the business behaves differently across groups.

This is one of the most important transitions in Topic 1:
**descriptive statistics become analytical interpretation.**

---

## Section 4 — Dispersion and Variability

Central tendency tells us the "centre." **Dispersion** tells us how spread out the data is.

Key measures:
- **Standard deviation (std)** — average distance from the mean.
- **Range** — difference between max and min.
- **Interquartile range (IQR)** — range of the middle 50 % of values.

In [ ]:
# Dispersion summary for invoice_amount
col = "invoice_amount"
print(f"Variable: {col}")
print(f"  Std dev : {df[col].std():8.1f}")
print(f"  Min     : {df[col].min():8.0f}")
print(f"  Max     : {df[col].max():8.0f}")
print(f"  Range   : {df[col].max() - df[col].min():8.0f}")
print(f"  Q1 (25%): {df[col].quantile(0.25):8.1f}")
print(f"  Q3 (75%): {df[col].quantile(0.75):8.1f}")
print(f"  IQR     : {df[col].quantile(0.75) - df[col].quantile(0.25):8.1f}")

In [ ]:
# Compare variability across customer segments
df.groupby("customer_segment")["invoice_amount"].agg(["std", "min", "max", "count"])

> **Interpretation:** a segment with high standard deviation relative to its mean has
> more heterogeneous invoicing behaviour. This could signal different business relationships
> or different product mixes within that segment.

---

## Section 5 — Distribution Shape

Numbers summarise; **charts reveal**. Let's look at the distributions visually.

In [ ]:
# Histogram 1 — invoice amount
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["invoice_amount"], bins=8, color="steelblue", edgecolor="white")
ax.set_xlabel("Invoice Amount (EUR)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Invoice Amounts")
ax.axvline(df["invoice_amount"].mean(), color="red", linestyle="--", label=f'Mean = {df["invoice_amount"].mean():.0f}')
ax.axvline(df["invoice_amount"].median(), color="orange", linestyle="--", label=f'Median = {df["invoice_amount"].median():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Histogram 2 — total amount
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["total_amount"], bins=8, color="darkorange", edgecolor="white")
ax.set_xlabel("Total Amount (EUR)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Total Amounts (incl. VAT)")
ax.axvline(df["total_amount"].mean(), color="red", linestyle="--", label=f'Mean = {df["total_amount"].mean():.0f}')
ax.axvline(df["total_amount"].median(), color="navy", linestyle="--", label=f'Median = {df["total_amount"].median():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

**Reading the histograms:**

- Is the distribution **symmetric** (bell-shaped) or **skewed** (tail on one side)?
- Are there values that look unusually far from the centre (**outliers**)?
- Does the shape suggest a few large invoices pulling the mean upward?

In [ ]:
# Box plot — invoice amount by customer segment
fig, ax = plt.subplots(figsize=(7, 4))
df.boxplot(column="invoice_amount", by="customer_segment", ax=ax,
           patch_artist=True,
           boxprops=dict(facecolor="lightblue"),
           medianprops=dict(color="red", linewidth=2))
ax.set_xlabel("Customer Segment")
ax.set_ylabel("Invoice Amount (EUR)")
ax.set_title("Invoice Amount by Customer Segment")
fig.suptitle("")  # remove automatic pandas title
plt.tight_layout()
plt.show()

> The box plot shows the **median** (red line), the **IQR** (the box), and potential
> outliers (dots beyond the whiskers). Compare how the spread differs across segments.

### Plots reveal structure that summary tables may hide

Histograms and box plots help students see shape, skewness, concentration and unusual values. A summary table may tell us the average. A plot tells us whether that average is being pulled by a few unusually large cases.

This matters because later modelling choices often depend on what this stage reveals.

---

## Section 6 — Segment Comparison

Descriptive statistics become much more useful when we compare **groups**.

In [ ]:
# Summary by customer segment
segment_summary = (
    df.groupby("customer_segment")
    .agg(
        n_invoices=("invoice_id", "count"),
        total_revenue=("total_amount", "sum"),
        avg_invoice=("invoice_amount", "mean"),
        pct_paid=("invoice_paid", lambda x: (x == "yes").mean() * 100),
    )
    .round(1)
)
segment_summary

In [ ]:
# Bar chart — total revenue by customer segment
fig, ax = plt.subplots(figsize=(7, 4))
segment_summary["total_revenue"].sort_values().plot.barh(ax=ax, color="teal", edgecolor="white")
ax.set_xlabel("Total Revenue (EUR)")
ax.set_title("Total Revenue by Customer Segment")
plt.tight_layout()
plt.show()

In [ ]:
# Summary by month
month_summary = (
    df.groupby("month_name")
    .agg(
        n_invoices=("invoice_id", "count"),
        avg_amount=("invoice_amount", "mean"),
        total_revenue=("total_amount", "sum"),
    )
    .round(1)
)
month_summary

> **Questions to consider:**
> - Which segment generates the most total revenue?
> - Is that because they have more invoices, higher amounts, or both?
> - Does the payment rate (`pct_paid`) vary across segments?

### Segment comparison prepares the modelling mindset

Comparing segments is already close to asking a modelling question. If one group behaves systematically differently from another, students should start wondering:

- would that variable be useful later?
- is the difference operationally meaningful?
- does it suggest a future feature or a future target relationship?

This is how EDA starts pointing toward model design without becoming modelling itself.

---

## Section 7 — Bivariate Exploration

So far we have looked at one variable at a time. Now let's examine **relationships between
two variables**.

> **Important:** correlation does **not** imply causation. Two variables can move together
> without one causing the other.

In [ ]:
# Scatter plot — invoice amount vs total amount
fig, ax = plt.subplots(figsize=(7, 5))

colors = {"customer_A": "steelblue", "customer_B": "darkorange",
          "customer_C": "seagreen", "customer_D": "crimson"}

for seg, group in df.groupby("customer_segment"):
    ax.scatter(group["invoice_amount"], group["total_amount"],
               label=seg, color=colors[seg], alpha=0.7, edgecolors="white", s=60)

ax.set_xlabel("Invoice Amount (EUR)")
ax.set_ylabel("Total Amount (EUR)")
ax.set_title("Invoice Amount vs Total Amount by Segment")
ax.legend(title="Segment")
plt.tight_layout()
plt.show()

The strong linear relationship is expected: `total_amount ≈ invoice_amount × (1 + vat_pct/100)`.
The two slightly separate "lanes" correspond to the two VAT rates (10 % vs 21 %).

This is a useful observation: **if we later build a model with `total_amount` as a feature,
`invoice_amount` would be nearly redundant** — they carry almost the same information.
This is called **multicollinearity**, and we will revisit it in later notebooks.

In [ ]:
# Correlation matrix (numeric columns only)
numeric_cols = ["invoice_amount", "vat_pct", "vat_amount", "total_amount"]
corr = df[numeric_cols].corr().round(2)
corr

> Look at the correlation table:
> - `invoice_amount` and `total_amount` are very highly correlated — as expected.
> - `vat_pct` and `vat_amount` have a moderate positive relationship.
> - For modelling purposes, keeping both `invoice_amount` and `total_amount` would be redundant.

### Correlation as an early warning sign

A strong relationship between two variables can be informative, but it can also be a warning.

If two variables are almost the same in analytical terms, keeping both later may add little information and may even create redundancy. In more advanced settings, a relationship this strong can also be a clue that some variables may become too close to the target or to one another.

This is why correlation is both an explanatory tool and an early design check.

---

## Section 8 — Your First EDA Narrative

A good analyst does not just compute numbers — they tell a **story**.

Below is a template for a short "analyst memo" summarising what we found. Fill in the blanks
(or write your own version in a new markdown cell):

---

### Mini Analyst Memo — Invoice Dataset

**What the dataset represents:**  
This dataset contains ___ invoices from ___ customer segments across ___ months, recording
invoice amounts, VAT details, and payment status.

**Key descriptive findings:**  
- The average invoice amount is approximately ___ EUR, with a median of ___ EUR.
- Customer segment ___ generates the highest total revenue.
- The payment rate across all invoices is about ___ %.

**Data quality notes:**  
- There are ___ missing values in `vat_amount`. These could be recalculated from
  `invoice_amount` and `vat_pct`.
- No duplicates or impossible values were detected.

**Variables that might matter for prediction:**  
- `invoice_amount` (strong driver of revenue)
- `customer_segment` (different behaviours across groups)
- `invoice_paid` (could serve as a target variable for a payment-prediction model)

**Open questions:**  
- Why are some VAT amounts missing? Is it a data-entry issue?
- Does the payment behaviour differ by invoice size or segment?

---

> Try writing your own version of this memo in a markdown cell below.

---

## Section 9 — Transition to the Modelling-Table Mindset

You now know how to **describe** a dataset. But description is not prediction.

Before running any machine-learning model, we first need to answer:

| Question | Why it matters |
|----------|----------------|
| **What is the target variable?** | The model needs a clear outcome to predict. |
| **What is one row?** | Each row must represent one independent observation. |
| **Which features go in?** | Not all columns should be features (e.g., IDs, redundant variables). |
| **How do we split the data?** | Train / validation / test sets prevent overfitting. |

In **Notebook 04** we will take a larger dataset and build a proper **modelling table** —
selecting a target, engineering features, and preparing the data for a baseline model.

---

### Self-practice tasks

1. Compute one additional grouped summary (e.g., mean `total_amount` by `vat_type`).
2. Identify one possible issue or pattern that we did not discuss in class.
3. Create one extra chart (e.g., a histogram of `vat_amount` for non-missing rows).
4. Write a 5-sentence EDA summary in business language, as if presenting to a manager.
5. Name 3 variables from this dataset that could enter a future prediction model and explain why.

---

*End of Notebook 03 — Descriptive Statistics and EDA for Finance*

## Bridge to Notebook 04

Notebook 03 closes with a descriptive story. Notebook 04 begins a design story.

Now that we have diagnosed the data, the next step is to formalise the business question into a modelling table (NB04).

After EDA, the questions are no longer:
- What does this dataset look like?
- Which segment is larger?
- Is the distribution skewed?

The next questions become:
- What is the unit of analysis?
- What is the target?
- Which rows should be eligible for modelling?
- Which variables are meaningful and available at decision time?

That transition — from diagnosis to analytical design — is exactly what Topic 2 needs.

---

**Project chain:** NB01 (setup) → NB02 (Python warmup) → **NB03 (EDA)** → NB04 (modelling table) → NB05 (baseline) → NB05b (trees) → NB06 (validate & interpret)  
**Current position:** NB03